In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import lsq_linear
import pandas as pd
from scipy.linalg import lstsq
from sklearn.model_selection import train_test_split
import cvxpy as cp

In [ ]:
def train(q_hp, delta_T_i, delta_T_a, q_solar, idx_night, delta_t):
    N = len(delta_T_i)
    
    # Expression
    X1 = delta_T_a*delta_t
    X2 = q_hp
    X3 = idx_night
    X4 = q_solar 
    
    Y_n = delta_T_i
     
    X_n = np.column_stack([X1, X2, X3, X4])
    lb = [0.0, 0.0, -np.inf, 0.0]
    ub = [ np.inf,  np.inf, np.inf, np.inf]

    res = lsq_linear(X_n, Y_n, bounds=(lb, ub))

    theta_n = res.x
    residuals_n = res.cost * 2   # lsq_linear reports 1/2 ||r||^2

    # recover parameters
    alpha, beta, gamma, epsilon = theta_n   # [1/(C·R), 1/C, w_n/C]
    C     = 1.0 / beta
    R_a   = beta / alpha
    w_n   = gamma * C
    eta_s   = epsilon * C
    rmse_n  = np.sqrt(residuals_n / len(Y_n))

    return C, R_a, w_n, eta_s, rmse_n

In [ ]:
def train_last_chance(q_hp, delta_T_i, delta_T_a, q_solar, idx_night, delta_t):
    N = len(delta_T_i)
    
    # Expression
    X1 = delta_T_a*delta_t
    X2 = q_hp
    X3 = idx_night
    X4 = q_solar 
    
    Y_n = delta_T_i
     
    X_n = np.column_stack([X1, X2, X3, X4])
    lb = [0.0, 0.0, -np.inf, -np.inf]
    ub = [ np.inf,  np.inf, np.inf, np.inf]

    res = lsq_linear(X_n, Y_n, bounds=(lb, ub))

    theta_n = res.x
    residuals_n = res.cost * 2   # lsq_linear reports 1/2 ||r||^2

    # recover parameters
    alpha, beta, gamma, epsilon = theta_n   # [1/(C·R), 1/C, w_n/C]
    C     = 1.0 / beta
    R_a   = beta / alpha
    w_n   = gamma * C
    eta_s   = epsilon * C
    rmse_n  = np.sqrt(residuals_n / len(Y_n))

    return C, R_a, w_n, eta_s, rmse_n

In [ ]:
def year_windows_from_first_ts(first_ts: pd.Timestamp):
    """
    Calendar-year window:
    - Year starts Jan 1 of first_ts.year
    - Training window is Jan 1 – Mar 31
    - Year ends Dec 31
    """
    year = first_ts.year + 1

    year_start = pd.Timestamp(year=year, month=1, day=1)
    train_end  = pd.Timestamp(year=year, month=3, day=31, hour=23, minute=59, second=59)
    year_end   = pd.Timestamp(year=year, month=12, day=31, hour=23, minute=59, second=59)

    return year_start, train_end, year_end


In [ ]:
def clean_and_gate_house_year(
    df_single: pd.DataFrame,
    t_start: pd.Timestamp,
    t_train_end: pd.Timestamp,
    t_year_end: pd.Timestamp,
    # columns to gate on
    col_Ti: str = "Internal_Air_Temperature",
    col_qhp: str = "Heat_Pump_Energy_Output",
    # sampling / interpolation behavior
    enforce_freq: str | None = "30min",   # set None to avoid forcing a grid
    max_gap_steps: int = 3,               # 4*30min = 2 hours
    do_interpolate: bool = True,
    # missingness thresholds (tune)
    max_missing_year: float = 0.90,       # allow up to 90% missing in year #
        # window
    max_missing_train: float = 0.20,      # allow up to 10% missing in 
        # training window
    # minimum data requirements (optional but useful)
    min_train_points: int = 1000,         # for 30-min data ~ 21 days; tune as needed
    verbose: bool = False,
):
    diag = {}

    # 0) Ensure datetime index
    if "Timestamp" in df_single.columns:
        df = df_single.copy()
        df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
        df = df.dropna(subset=["Timestamp"]).set_index("Timestamp").sort_index()
        df[col_qhp] = df[col_qhp].clip(lower=0)

    else:
        # already indexed by timestamp
        df = df_single.copy()
        df.index = pd.to_datetime(df.index, errors="coerce")
        df = df.dropna(subset=[col_Ti, col_qhp], how="all").sort_index()
        df[col_qhp] = df[col_qhp].clip(lower=0)


    # 1) Slice to chosen year window
    df_year = df.loc[(df.index >= t_start) & (df.index <= t_year_end)].copy()
    if df_year.empty:
        diag.update({"reason": "empty_year_window", "n_year_rows": 0})
        return None, None, diag

    # 2) Optionally regularize to a strict grid *after* slicing
    # (do this here so you don't manufacture NaNs outside the target window)
    if enforce_freq is not None:
        # round to grid, merge duplicates, then resample
        df_year.index = df_year.index.round(enforce_freq)
        df_year = df_year.groupby(df_year.index).mean(numeric_only=True)
        df_year = df_year.resample(enforce_freq).mean()

    # 3) Training slice (Jan–Mar of the selected year)
    df_train = df_year.loc[(df_year.index >= t_start) & (df_year.index <= t_train_end)].copy()
    if df_train.empty:
        diag.update({"reason": "no_data_in_training_window", "n_train_rows": 0, "n_year_rows": len(df_year)})
        return None, None, diag

    # 4) Missingness BEFORE interpolation (only on the gating columns)
    req_cols = [c for c in [col_Ti, col_qhp] if c in df_year.columns]
    if len(req_cols) < 2:
        diag.update({"reason": "missing_required_columns", "required_cols_present": req_cols})
        return None, None, diag


    miss_year_Ti = float(df_year[col_Ti].isna().mean())
    miss_year_hp = float(df_year[col_qhp].isna().mean())
    miss_train_Ti = float(df_train[col_Ti].isna().mean())
    miss_train_hp = float(df_train[col_qhp].isna().mean())



    diag.update({
        "n_year_rows": len(df_year),
        "n_train_rows": len(df_train),
        "miss_year_before": {col_Ti: miss_year_Ti, col_qhp: miss_year_hp},
        "miss_train_before": {col_Ti: miss_train_Ti, col_qhp: miss_train_hp},
    })

    # 5) Gate: if too much missing data, skip this house
    if (miss_train_Ti > max_missing_train) or (miss_train_hp > max_missing_train):
        diag["reason"] = "missingness_threshold_exceeded_train"
        return None, None, diag

    # 6) Interpolate short gaps (numeric only), then drop remaining NaNs in required cols
    df_filled = df_year.copy()
    if do_interpolate:
        num_cols = df_filled.select_dtypes(include=["number"]).columns

        for col in num_cols:
            s = df_filled[col]

            # length of each consecutive NaN run (per element)
            grp = s.isna().ne(s.isna().shift()).cumsum()
            run_len = s.isna().groupby(grp).transform("sum")

            # only NaNs in runs <= max_gap_steps are allowed to be filled
            allow_fill = s.isna() & (run_len <= max_gap_steps)

            # interpolate (fills all internal gaps), then keep fills ONLY where allowed
            s_interp = s.interpolate(method="time", limit_area="inside", limit_direction="forward")
            df_filled[col] = s_interp.where(allow_fill, s)

        '''
        df_filled[num_cols] = df_filled[num_cols].interpolate(
            method="time",
            limit=max_gap_steps,
            limit_direction="forward",
        )
        '''
    dropped_times = df_filled.index[df_filled[req_cols].isna().any(axis=1)]
    df_year_clean = df_filled.dropna(subset=req_cols).copy()
    df_train_clean = df_year_clean.loc[(df_year_clean.index >= t_start) & (df_year_clean.index <= t_train_end)].copy()



    diag.update({
        "n_year_rows_clean": len(df_year_clean),
        "n_train_rows_clean": len(df_train_clean),
        "miss_year_after": df_year_clean[req_cols].isna().mean().to_dict(),  # should be zeros
        "miss_train_after": df_train_clean[req_cols].isna().mean().to_dict(),
    })

    diag.update({
    "n_dropped_rows_req_cols": int(len(dropped_times)),
    "dropped_times_req_cols": dropped_times,   # DatetimeIndex
    })

    # 7) Minimum training points check (optional)
    if len(df_train_clean) < min_train_points:
        diag["reason"] = "too_few_training_points"
        return None, None, diag

    diag["reason"] = "ok"
    if verbose:
        print(diag)

    return df_year_clean, df_train_clean, diag

In [ ]:
df_train_q_exact = pd.read_parquet(
    "../retrieved_weather_data/q_streams_30min.parquet")

In [ ]:
df_train_q_exact

In [ ]:
#id_use = "EOH0005"
#df_id = df_train_q_exact[(id_use, "Q_hp_sc")]


In [ ]:
df_train_detached = pd.read_parquet(
    "../retrieved_weather_data/merged_data_final_homes.parquet")
type(df_train_detached)
df_train_detached

In [ ]:
unique_ids = df_train_detached["Property_ID"].unique()

df_house = pd.read_csv('../retrieved_weather_data/home_characteristics.csv')

# List of columns to check for missingness
cols = [
    "Bedrooms", "Floor_Height", "Habitable_Rooms", "House_Age",
    "House_Form", "No_Storeys", "No_Underfloor",
    "Total_Floor_Area", "Wall_Type", "MCS_DHWAnnual","HP_Size_kW",
    "HP_Installed", "House_SAP"
]

df_house_train = {}
for id_use in unique_ids:
    df_house_train[id_use] = df_house[df_house["Property_ID"] == id_use]

In [ ]:
missing_counts = df_house[df_house["Property_ID"].isin(unique_ids)][["Floor_Height", "Total_Floor_Area"]].isna().sum()
print(missing_counts)


In [ ]:
df_single = df_train_detached[
        df_train_detached["Property_ID"] == id_use].copy()

range_df_hp = (df_single .dropna(subset=["Heat_Pump_Energy_Output"]) .groupby("Property_ID")["Timestamp"] .agg(first_available="min", last_available="max") )

In [ ]:
df_single

In [ ]:
trained_params = pd.DataFrame(index=["Floor Area", "No_Storeys", "Wall_Type",
                                     "HP_Type", "House_SAP",
                                     "C", "R_a", "w_s", "w_n",
                                     "rmse", "train_start", "train_end",
                                     "val_start", "val_end", "missing_data"])

req_cols = ["Heat_Pump_Energy_Output_Diff", "Internal_Temperature_Diff",
            "Internal_Ambient_Temperature_Diff", ]

for id_use in unique_ids:
    wall = df_house_train[id_use].Wall_Type.to_string()
    storeys = df_house_train[id_use].No_Storeys.to_string()
    floor_area = df_house_train[id_use].Total_Floor_Area.to_string()
    hp_type = df_house_train[id_use].HP_Installed.to_string()
    house_sap = df_house_train[id_use].House_SAP.to_string()

    df_q_exact_single = df_train_q_exact[id_use]

    df_single = df_train_detached[
        df_train_detached["Property_ID"] == id_use].copy()

    df_single["Heat_Pump_Energy_Output_Diff"] = df_single[
        "Heat_Pump_Energy_Output"].diff()
    df_single["Internal_Temperature_Diff"] = df_single[
        "Internal_Air_Temperature"].diff()
    df_single["Internal_Ambient_Temperature_Diff"] = \
        (df_single["temp"] -
         df_single["Internal_Air_Temperature"])
    df_single["Immersion_Diff"] = df_single[
        "Immersion_Heater_Energy_Consumed"].diff()

    range_df_hp = (df_single.dropna(subset=["Heat_Pump_Energy_Output"]) .groupby("Property_ID")["Timestamp"] .agg(first_available="min", last_available="max") )
    first_ts = range_df_hp["first_available"].iloc[0]
    t_start, t_train_end, t_year_end = year_windows_from_first_ts(first_ts)


    #df_year = df_single[(df_single["Timestamp"] >= t_start) & ( # df_single["Timestamp"] <= t_year_end)] #df_train_3mo = df_single[(df_single["Timestamp"] >= t_start) & ( # df_single["Timestamp"] <= t_train_end)] #df_val_rest = df_single[(df_single["Timestamp"] > t_train_end) & ( # df_single["Timestamp"] <= t_year_end)]
    t_start = t_start - pd.DateOffset(months=3)
    t_train_end = t_train_end - pd.DateOffset(months=3)
    t_start_val = t_start + pd.DateOffset(months=3)
    t_end_val = t_start_val + pd.DateOffset(months=1)
    df_year_clean, df_train_clean, diag = clean_and_gate_house_year(
        df_single=df_single,
        t_start=t_start,
        t_train_end=t_train_end,
        t_year_end=t_year_end,
        col_Ti="Internal_Air_Temperature",
        col_qhp="Heat_Pump_Energy_Output_Diff",
        enforce_freq="30min",
        max_gap_steps=6,
        do_interpolate=True,
        max_missing_year=0.20,    # <-- tune
        max_missing_train=0.10,   # <-- tune
        min_train_points=1000,    # <-- tune
        verbose=False,
        )

    if df_year_clean is None:
        print("Trying 2")
        t_start = t_start + pd.DateOffset(months=3)
        t_train_end = t_train_end + pd.DateOffset(months=3)
        t_start_val = t_start + pd.DateOffset(months=3)
        t_end_val = t_start_val + pd.DateOffset(months=3)
        df_year_clean, df_train_clean, diag = clean_and_gate_house_year(
            df_single=df_single,
            t_start=t_start,
            t_train_end=t_train_end,
            t_year_end=t_year_end,
            col_Ti="Internal_Air_Temperature",
            col_qhp="Heat_Pump_Energy_Output",
            enforce_freq="30min",
            max_gap_steps=6,
            do_interpolate=True,
            max_missing_year=0.20,    # <-- tune
            max_missing_train=0.20,   # <-- tune
            min_train_points=1000,    # <-- tune
            verbose=False,
            )

    if df_year_clean is None:
        print("Trying 3, more relaxed missing data")
        df_year_clean, df_train_clean, diag = clean_and_gate_house_year(
            df_single=df_single,
            t_start=t_start,
            t_train_end=t_train_end,
            t_year_end=t_year_end,
            col_Ti="Internal_Air_Temperature",
            col_qhp="Heat_Pump_Energy_Output",
            enforce_freq="30min",
            max_gap_steps=6,
            do_interpolate=True,
            max_missing_year=0.50,    # <-- tune
            max_missing_train=0.40,   # <-- tune
            min_train_points=1000,    # <-- tune
            verbose=False,
            )

    if df_year_clean is None:
        # house unusable -> move to next
        # (optional) print why:
        print(f"skip {id_use}: {diag['reason']} | miss_year={diag.get('miss_year_before')}")
        continue

    df_year_clean = df_year_clean.dropna(subset=["Internal_Temperature_Diff"])
    # ------ Prepare Training Data ----------
    df_heating_single = df_year_clean.copy()

    df_heating_annual = df_heating_single[df_heating_single.index >= t_start]
    df_heating_annual = df_heating_annual[df_heating_annual.index <= t_train_end]

    df_30min_val = df_q_exact_single[df_q_exact_single.index >= t_start]
    df_30min_val = df_30min_val[df_30min_val.index <= t_train_end]

        # Extract hour from Timestamp
    df_heating_annual_night = df_heating_annual.copy()
    df_heating_annual_night['Hour'] = df_heating_annual.index.hour

    df_heating_annual = df_heating_annual.sort_index()
    df_30min_val = (df_30min_val
                    .sort_index()
                    .dropna(subset=["Q_hp_total"])
                    )


    common_idx = df_heating_annual_night.index.intersection(df_30min_val.index)
    df_heating_annual_night = df_heating_annual_night.loc[common_idx]
    df_30min_val = df_30min_val.loc[common_idx]

    # Define night and day conditions
    # Condition 1: Time is between 10 PM and 6 AM
    night_time_condition = (df_heating_annual_night['Hour'] >= 20) | \
                           (df_heating_annual_night['Hour'] < 6)

    # Condition 2: Solar Irradiation is 0 (assuming 'SolarRadiation' is the correct column name)
    if 'solarradiation' in df_heating_annual_night.columns:
        night_solar_condition = (df_heating_annual_night['solarradiation'] == 0)
        df_heating_annual_night['is_night'] = night_time_condition & night_solar_condition
    else:
        print("\nWarning: 'SolarRadiation' column not found after preprocessing. Cannot use it for night/day split.")
        df_heating_annual_night['is_night'] = night_time_condition # Fallback to just time condition


    t_step = 30  # minutes
    delta_t = t_step / 60

    #Night data

    T_a = (df_heating_annual_night["temp"].iloc[:-1]
              .reset_index
           (drop=True).to_numpy())
    T_i = (df_heating_annual_night["Internal_Air_Temperature"].iloc[:-1].reset_index
           (drop=True).to_numpy())
    delta_T_a = (
        df_heating_annual_night["Internal_Ambient_Temperature_Diff"].iloc[:-1]
        .reset_index(drop=True).to_numpy())
    delta_T_i = (df_heating_annual_night["Internal_Temperature_Diff"].iloc[1:]
                 .reset_index(drop=True).to_numpy() / delta_t)
    q_hp = (df_30min_val["Q_hp_sc"].iloc[:-1]
            .reset_index(drop=True).to_numpy())
    q_solar = (df_heating_annual_night["solarradiation"].iloc[:-1].reset_index
               (drop=True).to_numpy())
    v_wind = (df_heating_annual_night["windspeed"].iloc[:-1].reset_index
              (drop=True)
              .to_numpy())
    q_imm = (df_heating_annual_night["Immersion_Diff"].iloc[:-1]
            .reset_index(drop=True).to_numpy())

    #q_tot = q_hp+q_imm
    idx_night = df_heating_annual_night['is_night'].astype(int).iloc[:-1].to_numpy()

    # Create time index for in the loop q_hat arrays
    time_index = df_heating_annual.index[:-1]
    df_q_results = pd.DataFrame(index=time_index)

    missing_data = diag["miss_train_before"]

    vars_to_check = {
        "q_hp": q_hp,
        "delta_T_i": delta_T_i,
        "delta_T_a": delta_T_a,
        "q_solar": q_solar,
        "idx_night": idx_night,
        "delta_t": delta_t,
    }

    bad = {
        k: v for k, v in vars_to_check.items()
        if v is None or (
            hasattr(v, "__len__") and pd.isna(v).any()
        ) or (
            not hasattr(v, "__len__") and pd.isna(v)
        )
    }

    if bad:
        raise ValueError(f"NaN/None detected in inputs: {list(bad.keys())}")

    try:
        # Primary training attempt
        C, R_a, w_n, eta_s, rmse = train(
            q_hp, delta_T_i, delta_T_a, q_solar,
            idx_night, delta_t
        )

    except Exception as e1:
        try:
            # Fallback / last-chance attempt
            C, R_a, w_n, eta_s, rmse = train_last_chance(
                q_hp, delta_T_i, delta_T_a, q_solar,
                idx_night, delta_t
            )

        except Exception as e2:
            print(f"Training failed for {id_use}: primary={e1}, fallback={e2}")
            continue

    w_s = eta_s

    trained_params[id_use] = [floor_area, storeys, wall, hp_type, house_sap,
                              C,R_a,eta_s,w_n,rmse, t_start, t_train_end,
                              t_start_val, t_end_val, missing_data]
    '''
    j=1
    if j == 1:
        break
        stop
    '''


In [ ]:
plt.plot(df_heating_annual_night["External_Air_Temperature"].ffill(),
         label="External"
                                                                    " Air Temperature")
plt.plot(df_heating_annual_night["temp"], label="temp")
plt.legend()

In [ ]:
trained_params.to_csv("trained_params_lin_regres_disag_v1.csv", index=True)
trained_params


In [ ]:
miss = trained_params.loc["R_a"].isna().sum()
tot = trained_params.shape[1]

tot-miss

In [ ]:
trained_params

In [ ]:
unique_ids_trained = trained_params.columns.to_list()

In [ ]:
df_single

In [ ]:
id_use

In [ ]:
# ------- Look at heat supply performance in validation data ------
for id_use in unique_ids_trained:
    df_id = trained_params[id_use].copy()
    C = df_id.C
    R_a = df_id.R_a
    w_s= df_id.w_s
    w_n = df_id.w_n
    t_start_val = df_id.val_start
    t_end_val = df_id.val_end
    t_start_train = df_id.train_start
    t_end_train = df_id.train_end


    df_q_exact_single = df_train_q_exact[id_use]
    
    df_single = df_train_detached[df_train_detached["Property_ID"] == id_use].copy()
    
    #re-adjust Heat Pump Diff and add temp differences
    df_single["Heat_Pump_Energy_Output_Diff"] = df_single["Heat_Pump_Energy_Output"].diff()
    df_single["Internal_Temperature_Diff"] = df_single["Internal_Air_Temperature"].diff()
    df_single["Internal_Ambient_Temperature_Diff"] = \
        (df_single["temp"] -
         df_single["Internal_Air_Temperature"])
    
    # 1. Drop columns with almost all missing data (e.g., more than 90% missing)
    threshold = 0.20 * len(df_single)
    df_single_cleaned = df_single.dropna(axis=1, thresh=threshold)
    #print("Columns dropped due to high missing values:")
    #print(df_single.columns.difference(df_single_cleaned.columns).tolist())
    
    df_single = df_single_cleaned
    
    #print("\nColumns remaining after dropping highly missing columns:")
    #print(df_single.columns.tolist())
    
    # 2. Handle missing values: Interpolate if missing for up to 2 hours (4 half-hour intervals), else drop rows
    df_single = df_single.set_index('Timestamp')
    df_single = df_single.sort_index()
    
    # Apply interpolation with a limit of 4 (for 2 hours of half-hourly data)
    numeric_cols = df_single.select_dtypes(include=['number']).columns
    df_single_numeric_interpolated = df_single[numeric_cols].interpolate(method='time', limit=4, limit_direction='both')
    
    df_single_interpolated = df_single.copy() 
    df_single_interpolated[numeric_cols] = df_single_numeric_interpolated
    
    # After interpolation, drop rows that still contain NaN values (meaning they were missing for > 2 hours)
    initial_rows = len(df_single_interpolated)
    df_single_processed = df_single_interpolated.dropna()
    rows_dropped_after_interpolation = initial_rows - len(df_single_processed)
    
    #print(f"\nNumber of rows dropped after handling NA values (missing for >
    # 2 hours): {rows_dropped_after_interpolation}")
    
    # -----EXTRACT SUMMER DATA -------# 
    df_heating_single = df_single_processed.copy()
    
    t_start = t_start_val
    t_end =  df_id.val_end
    df_q_opt = df_q_results[(df_q_results.index>=t_start) & (df_q_results.index<=t_end)]
    df_heating_val = df_heating_single[df_heating_single.index>=t_start]
    df_heating_val = df_heating_val[df_heating_val.index<=t_end]


    df_30min_val = df_q_exact_single[df_q_exact_single.index >= t_start]
    df_30min_val = df_30min_val[df_30min_val.index <= t_end]

    #df_heating_annual = df_heating_annual[
    #    (df_heating_annual.index <= t_mid_end) | (df_heating_annual
    #                                              .index >= t_mid_start)]

        # Extract hour from Timestamp
    df_heating_annual_night = df_heating_annual.copy()
    df_heating_annual_night['Hour'] = df_heating_annual.index.hour
    
    # make sure you have an Hour column
    df_heating_val = df_heating_val.assign(Hour=df_heating_val.index.hour)
    
    # define night‑time & zero‑solar conditions
    night_time = (df_heating_val['Hour'] >= 22) | (df_heating_val['Hour'] < 6)
    zero_solar = (df_heating_val['solarradiation'] == 0)
    
    # vectorized assignment of w
    df_heating_val['w'] = np.where(night_time & zero_solar, w_n, 0.0)

    common_idx = df_heating_val.index.intersection(df_30min_val.index)
    df_heating_val = df_heating_val.loc[common_idx]
    df_30min_val = df_30min_val.loc[common_idx]
    
    w = df_heating_val['w'].iloc[:-1].reset_index(drop=True).to_numpy()
    
    T_a_val = (df_heating_val["temp"].iloc[:-1].reset_index
           (drop=True).to_numpy()) 
    T_i_val = (df_heating_val["Internal_Air_Temperature"].iloc[:-1].reset_index
           (drop=True).to_numpy()) 
    delta_T_a_val = (df_heating_val["Internal_Ambient_Temperature_Diff"].iloc[:-1]
                 .reset_index(drop=True).to_numpy()) 
    delta_T_i_val = (df_heating_val["Internal_Temperature_Diff"].iloc[1:]
                 .reset_index(drop=True).to_numpy() / delta_t) 
    q_hp_val = (df_30min_val["Q_hp_sc"].iloc[:-1]
            .reset_index(drop=True).to_numpy()) 
    q_solar_val = (df_heating_val["solarradiation"].iloc[:-1].reset_index
               (drop=True).to_numpy()) 
    v_wind_val = (df_heating_val["windspeed"].iloc[:-1].reset_index(drop=True)
              .to_numpy()) 
    
    
    
    q_hat_sim = (delta_T_i_val * C -  delta_T_a_val / R_a * delta_t  - w_s *  
            q_solar_val  - w * np.ones(len(T_a_val), )* delta_t )
    
    e_q = q_hat_sim - q_hp_val
    rmse_q_val = np.sqrt(np.mean(e_q**2))
    print(f"RMSE of q_hp: {rmse_q_val}")
    
    plt.plot(df_heating_val.index[1:],q_hat_sim, label = "Sim")
    plt.plot(df_heating_val.index[1:], q_hp_val, label = "Measured")
    plt.legend()
    plt.xlabel("Time")
    plt.ylabel("kWh")
    plt.title(f"{id_use}")
    plt.show()

In [ ]:
len(df_30min_val)

In [ ]:
df_heating_annual

In [ ]:
df_30min_val

In [ ]:
df_heating_val

In [ ]:
df_id

In [ ]:
df_30min_val = df_q_exact_single[df_q_exact_single.index >= t_start]
df_30min_val = df_30min_val[df_30min_val.index <= t_end]
df_30min_val

In [ ]:
# -----Look at temperature tracking performance in the training --------------

for id_use in unique_ids_trained:
    df_id = trained_params[id_use].copy()
    C = df_id.C
    R_a = df_id.R_a
    w_s = df_id.w_s
    w_n = df_id.w_n
    t_start_val = df_id.val_start
    t_end_val = df_id.val_end
    t_start_train = df_id.train_start
    t_end_train = df_id.train_end

    #df_single_2min = df_train_detached_2min.loc[id_use]
    #df_single_2min["ETotal_step_hp"] =
    # df_single_2min["Heat_Pump_Energy_Output"].diff()
    #df_single_2min["ETotal_step_im"] =
    # df_single_2min["Immersion_Heater_Energy_Consumed"].diff()

    df_q_exact_single = df_train_q_exact[id_use]

    df_single = df_train_detached[df_train_detached["Property_ID"] == id_use].copy()
    
    #re-adjust Heat Pump Diff and add temp differences
    df_single["Heat_Pump_Energy_Output_Diff"] = df_single["Heat_Pump_Energy_Output"].diff()
    df_single["Internal_Temperature_Diff"] = df_single["Internal_Air_Temperature"].diff()
    df_single["Internal_Ambient_Temperature_Diff"] = \
        (df_single["External_Air_Temperature"] - 
         df_single["Internal_Air_Temperature"])
         
    
    
    # 1. Drop columns with almost all missing data (e.g., more than 90% missing)
    threshold = 0.20 * len(df_single)
    df_single_cleaned = df_single.dropna(axis=1, thresh=threshold)
    #print("Columns dropped due to high missing values:")
    #print(df_single.columns.difference(df_single_cleaned.columns).tolist())
    
    df_single = df_single_cleaned
    
    #print("\nColumns remaining after dropping highly missing columns:")
    #print(df_single.columns.tolist())
    
    # 2. Handle missing values: Interpolate if missing for up to 2 hours (4 half-hour intervals), else drop rows
    df_single = df_single.set_index('Timestamp')
    df_single = df_single.sort_index()
    
    # Apply interpolation with a limit of 4 (for 2 hours of half-hourly data)
    numeric_cols = df_single.select_dtypes(include=['number']).columns
    df_single_numeric_interpolated = df_single[numeric_cols].interpolate(method='time', limit=4, limit_direction='both')
    
    df_single_interpolated = df_single.copy() 
    df_single_interpolated[numeric_cols] = df_single_numeric_interpolated
    
    # After interpolation, drop rows that still contain NaN values (meaning they were missing for > 2 hours)
    initial_rows = len(df_single_interpolated)
    df_single_processed = df_single_interpolated.dropna()
    rows_dropped_after_interpolation = initial_rows - len(df_single_processed)
    
    #print(f"\nNumber of rows dropped after handling NA values (missing for >
    # 2 hours): {rows_dropped_after_interpolation}")
    
    # -----EXTRACT SUMMER DATA -------# 
    df_heating_single = df_single_processed.copy()
    
    # Validate results
    t_start = t_start_val
    t_end = t_end_val
    df_q_id_use = df_q_results[(df_q_results.index>=t_start) & (df_q_results.index<=t_end)]
    df_heating_val = df_heating_single[df_heating_single.index>=t_start]
    df_heating_val = df_heating_val[df_heating_val.index<=t_end]

    df_30min_val = df_q_exact_single[df_q_exact_single.index >= t_start]
    df_30min_val = df_30min_val[df_30min_val.index <= t_end]

    #df_heating_annual = df_heating_annual[
    #    (df_heating_annual.index <= t_mid_end) | (df_heating_annual
    #                                              .index >= t_mid_start)]

        # Extract hour from Timestamp
    df_heating_annual_night = df_heating_annual.copy()
    df_heating_annual_night['Hour'] = df_heating_annual.index.hour

    # make sure you have an Hour column
    df_heating_val = df_heating_val.assign(Hour=df_heating_val.index.hour)
    
    # define night‑time & zero‑solar conditions
    night_time = (df_heating_val['Hour'] >= 22) | (df_heating_val['Hour'] < 6)
    zero_solar = (df_heating_val['solarradiation'] == 0)
    
    # vectorized assignment of w
    df_heating_val['w'] = np.where(night_time & zero_solar, w_n, 0.0)

    common_idx = df_heating_val.index.intersection(df_30min_val.index)
    df_heating_val = df_heating_val.loc[common_idx]
    df_30min_val = df_30min_val.loc[common_idx]
    
    w = df_heating_val['w'].iloc[:-1].reset_index(drop=True).to_numpy()
    
    T_a_val = (df_heating_val["temp"].iloc[:-1].reset_index
           (drop=True).to_numpy()) 
    T_i_val = (df_heating_val["Internal_Air_Temperature"].iloc[:-1].reset_index
           (drop=True).to_numpy()) 
    delta_T_a_val = (df_heating_val["Internal_Ambient_Temperature_Diff"].iloc[:-1]
                 .reset_index(drop=True).to_numpy()) 
    delta_T_i_val = (df_heating_val["Internal_Temperature_Diff"].iloc[1:]
                 .reset_index(drop=True).to_numpy() / delta_t) 
    q_hp_val = (df_30min_val["Q_hp_sc"].iloc[:-1]
            .reset_index(drop=True).to_numpy()) 
    q_solar_val = (df_heating_val["solarradiation"].iloc[:-1].reset_index
               (drop=True).to_numpy()) 
    v_wind_val = (df_heating_val["windspeed"].iloc[:-1].reset_index(drop=True)
              .to_numpy()) 
    
    
    delta_T_i_sim = (delta_T_a_val / R_a *delta_t + q_hp_val + w_s* q_solar_val + w
                     * np
                     .ones(len(T_a_val), )* delta_t) / C
    
    # --- align time index with arrays ---
    time_index = df_heating_val.index[:-1]   # aligns with T_i_val, delta_T_i_sim

    T_i_sim = np.zeros_like(T_i_val)

    # initialize from first measurement
    T_prev = T_i_val[0]

    for i in range(len(T_i_val)):
        ts = time_index[i]

        # reset at midnight (00:00)
        if ts.hour == 0 and ts.minute == 0:
            T_prev = T_i_val[i]

        T_i_sim[i] = T_prev + delta_T_i_sim[i] * delta_t
        T_prev = T_i_sim[i]
    
    
    plt.plot(df_heating_val.index[1:],delta_T_i_sim, label = "Sim")
    plt.plot(df_heating_val.index[1:], delta_T_i_val, label = "Measured")
    plt.legend()
    plt.xlabel("Time")
    plt.ylabel("Delta T")
    plt.title(f"{id_use}")
    plt.show()
    
    plt.plot(df_heating_val.index[1:],T_i_sim, label = "Sim")
    plt.plot(df_heating_val.index[1:], df_heating_val["Internal_Air_Temperature"]
             .iloc[1:].reset_index
           (drop=True).to_numpy(), label = "Measured")
    plt.legend()
    plt.xlabel("Time")
    plt.ylabel("Celsius")
    plt.title(f"{id_use}")
    plt.show()